<a href="https://colab.research.google.com/github/kph4br/ds2002-fa26/blob/main/studio/2026-10-09-Public-API-to-Clean-Table-Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)
        except requests.RequestException as err:
            last = type(err).__name__
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                raise ValueError(f'status {r.status_code}: {r.text[:200]}')
            last = f'status {r.status_code}'
        if attempt < tries - 1:
            delay = 2 ** attempt
            print(f'attempt {attempt + 1} failed ({last}). waiting {delay}s')
            time.sleep(delay)
    raise RuntimeError(f'gave up after {tries} attempts: {last}')

Finished function above. It has a timeout on every request, retries 429 and 5xx with a wait that doubles each time (1s, then 2s), and gives up with a RuntimeError if every try fails. A 400 or 404 raises a ValueError because that means my request is wrong, and retrying won't fix it. I used two different error types to make my life easier in Part 2 so I can tell “the server is down” apart from “I made a mistake.”

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
START, END = '2026-09-05', '2026-10-03'
HOME_DATES = [START, END]
LAT, LON = 38.03, -78.51
fetch_error = None

try:
    full = get_weather(START, END, lat=LAT, lon=LON)
    wx = full[full['time'].isin(HOME_DATES)].reset_index(drop=True)
    print(f'pulled {len(full)} days, kept {len(wx)}')
except RuntimeError as err:
    fetch_error = err
    print('FETCH FAILED:', err)

pulled 29 days, kept 2


I asked for one range from 9/5 to 10/3 so it's one request instead of two, then kept the two home dates with isin. If the API is down and every retry fails, I'll save the error and use the fallback in Part 3. If the problem is my own code though, (like a typo in the request) I'll let it crash instead, because using fake data there would hide my mistake. My call did work and it pulled 29 days and kept the 2 I need.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
if fetch_error is not None:
    used_fallback = True
    wx = FALLBACK.copy()
    print('USING FALLBACK SAMPLE DATA, not real weather')

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'
print(wx)

         time  temperature_2m_max  precipitation_sum      source
0  2026-09-05                29.4                2.5  open-meteo
1  2026-10-03                22.0               14.7  open-meteo


The fallback only gets used if the fetch actually failed, and when it does it prints a message and sets used_fallback = True. Every row gets a source column so anyone looking at the table can tell if the numbers are real or sample data. My run used open-meteo, so my data is real.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
RENAME = {'time': 'date',
          'temperature_2m_max': 'temp_max',
          'precipitation_sum': 'precip_mm'}

wx = wx.rename(columns=RENAME)
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]
print(wx)
print(wx.dtypes)

        date  temp_max  precip_mm      source
0 2026-09-05      29.4        2.5  open-meteo
1 2026-10-03      22.0       14.7  open-meteo
date         datetime64[ns]
temp_max            float64
precip_mm           float64
source               object
dtype: object


Renamed the API's columns. I converted date to a real datetime because it comes back as a string, and a string date won't line up with a datetime date when I join it to sales.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]
diff = wetter['precip_mm'] - drier['precip_mm']
note = ' (sample data, not real weather)' if used_fallback else ''

if diff == 0:
    print(f"Both home weekends got the same rain: {wetter['precip_mm']:.1f} mm{note}.")
else:
    print(f"{wetter['date'].strftime('%B %d, %Y')} was wetter, with {wetter['precip_mm']:.1f} mm of rain "
          f"compared to {drier['precip_mm']:.1f} mm on {drier['date'].strftime('%B %d, %Y')}, "
          f"so {diff:.1f} mm more{note}.")

October 03, 2026 was wetter, with 14.7 mm of rain compared to 2.5 mm on September 05, 2026, so 12.2 mm more.


October 3 was the wetter home weekend by 12.2 mm (14.7 mm vs 2.5 mm on September 5).

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
from datetime import datetime

print('Source:     ', 'fallback sample (NOT real data)' if used_fallback
      else 'Open-Meteo historical archive API')
print('Endpoint:   ', BASE)
print('Coordinates:', LAT, LON)
print('Date range: ', START, 'to', END)
print('Pulled:     ', datetime.now().strftime('%Y-%m-%d %H:%M'))

Source:      Open-Meteo historical archive API
Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03 -78.51
Date range:  2026-09-05 to 2026-10-03
Pulled:      2026-10-11 00:52


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame instead of raising, nothing would look wrong at first because there's no error. When I left-merge it onto 200 days of sales, every row keeps its revenue but the rain and temperature columns are all NaN. If I did an inner join instead, I'd end up with 0 rows. Then when I chart rain against revenue, pandas would skip the NaN points so the chart is empty, or if anyone filled the NaNs with 0 it shows every day as dry. A chart like that would make it look like rain has no effect on sales. Nobody would catch it because the code never crashes, the chart still renders, and “rain doesn't matter” sounds like a believable finding.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
CITIES = {'Charlottesville': (LAT, LON), 'Orlando': (28.54, -81.38)}
frames = []
for city, (lat, lon) in CITIES.items():
    city_wx = get_weather(START, END, lat=lat, lon=lon)
    city_wx = city_wx[city_wx['time'].isin(HOME_DATES)].rename(columns=RENAME)
    city_wx['date'] = pd.to_datetime(city_wx['date'])
    city_wx['city'] = city
    frames.append(city_wx)

both = pd.concat(frames, ignore_index=True)[['city', 'date', 'temp_max', 'precip_mm']]
print(both)
assert len(both) == 4

              city       date  temp_max  precip_mm
0  Charlottesville 2026-09-05      29.4        2.5
1  Charlottesville 2026-10-03      22.0       14.7
2          Orlando 2026-09-05      32.1        3.9
3          Orlando 2026-10-03      30.4       14.5
